# Notebook 2 — Per-90 정규화

`data/season_all.csv`를 로드하여 모든 카운트 스탯을 90분 기준으로 정규화합니다.

**입력:** `data/season_all.csv`  
**출력:** `data/stats.csv`

In [ ]:
import pandas as pd
import numpy as np
import os

DATA_DIR = 'data'
df = pd.read_csv(os.path.join(DATA_DIR, 'season_all.csv'))
print(f'로드 완료: {df.shape}')
df.head(3)

## 출전 시간 컬럼 확인

In [ ]:
# Notebook 1에서 저장한 컬럼명 중 분(minute) 컬럼을 자동 탐지
min_col = None
for c in df.columns:
    if c.lower() in ('min', 'minutes', 'playing_time_min', 'playing time_min'):
        min_col = c
        break

if min_col is None:
    print('출전 시간 컬럼 후보 (수동 지정 필요):')
    print([c for c in df.columns if 'min' in c.lower() or 'time' in c.lower()])
    raise ValueError("아래 셀에서 min_col = '컬럼명' 으로 직접 지정하세요")
else:
    print(f'출전 시간 컬럼: {min_col}')
    print(df[min_col].describe())

In [ ]:
# ★ 자동 탐지 실패 시 여기서 직접 지정
# min_col = 'Min'

## 정규화 제외 컬럼 정의

In [ ]:
# 메타 컬럼
META_COLS = ['League', 'Season', 'Team', 'Player', 'Pos', 'Nation', 'Age', 'Born', min_col]
META_COLS = [c for c in META_COLS if c in df.columns]

# 비율/퍼센트/이미 per-90인 컬럼 — 정규화 불필요
RATE_KEYWORDS = ['%', 'pct', '/90', 'per_90', 'per90', 'avg', 'ratio', '90s']
rate_cols = [c for c in df.columns
             if any(kw.lower() in c.lower() for kw in RATE_KEYWORDS)
             and c not in META_COLS]

EXCLUDE_COLS = set(META_COLS + rate_cols)

# 정규화 대상: 숫자이면서 제외 대상이 아닌 것
num_cols = df.select_dtypes(include='number').columns.tolist()
NORM_COLS = [c for c in num_cols if c not in EXCLUDE_COLS]

print(f'메타 컬럼       : {len(META_COLS)}개 → {META_COLS}')
print(f'비율 컬럼(제외) : {len(rate_cols)}개')
print(f'정규화 대상     : {len(NORM_COLS)}개')
print('\n정규화 대상 컬럼:')
print(NORM_COLS)

## Per-90 정규화 수행

In [ ]:
stats = df.copy()

minutes = stats[min_col].replace(0, np.nan)
for col in NORM_COLS:
    stats[col] = (stats[col] / minutes * 90).round(4)

stats[NORM_COLS] = stats[NORM_COLS].fillna(0)
print('정규화 완료')
stats[NORM_COLS[:5]].describe().round(3)

## Player ID 생성

`{League}_{Season}_{Player}` 형식 (예: `EPL_2021-22_Son Heung-min`)

In [ ]:
LEAGUE_SHORT = {
    'ENG-Premier League': 'EPL',
    'ESP-La Liga':        'LaLiga',
    'GER-Bundesliga':     'Bundesliga',
    'ITA-Serie A':        'SerieA',
}

league_short = stats['League'].map(LEAGUE_SHORT).fillna(stats['League'])
stats.insert(0, 'ID', league_short + '_' + stats['Season'].astype(str) + '_' + stats['Player'].astype(str))

# 이적 선수(같은 ID가 2행) → 마지막 행(시즌 합산본) 유지
dup = stats['ID'].duplicated().sum()
if dup > 0:
    print(f'중복 ID {dup}개 처리 (이적 선수)')
    stats = stats.drop_duplicates(subset='ID', keep='last')

print(f'최종 선수 수: {len(stats):,}')
print(f'ID 예시: {stats["ID"].iloc[:3].tolist()}')

## 저장

In [ ]:
out_path = os.path.join(DATA_DIR, 'stats.csv')
stats.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f'저장 완료: {out_path}')
print(f'  행: {len(stats):,}  |  컬럼: {len(stats.columns)}')
print(f'  NaN 잔여: {stats.isnull().sum().sum()}')
stats.head(3)